<div style="text-align: right;">
  <img src="https://raw.githubusercontent.com/exasol/ai-lab/refs/heads/main/assets/Exasol_Logo_2025_Dark.svg" style="width:200px; margin: 10px;" />
</div>

# Question answering with text generation model

In this notebook, we will load and use a text-generation language models that can retrieve the answer to a question from a given text.  Learn more about the Text Generation task <a href="https://huggingface.co/tasks/text-generation" target="_blank" rel="noopener">here</a>. Please also refer to the Transformer Extension <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a> to find more information about the UDFs used in this notebook.

We will be running SQL queries using <a href="https://github.com/ploomber/jupysql" target="_blank" rel="noopener">JupySQL</a>SQL Magic.

## Prerequisites

Prior to using this notebook the following steps need to be completed:
1. [Configure the AI Lab](../main_config.ipynb).
2. [Initialize the Transformer Extension](te_init.ipynb).

## Setup

### Open Secure Configuration Storage

In [ ]:
from exasol.nb_connector.ui.access import access_store
display(access_store.get_access_store('../'))

Let's bring up JupySQL and connect to the database via SQLAlchemy. Please refer to the documentation of <a href="https://github.com/exasol/sqlalchemy-exasol" target="_blank" rel="noopener">sqlalchemy-exasol</a> for details on how to connect to the database using the Exasol SQLAlchemy driver.

In [ ]:
from exasol.nb_connector.ui.common import jupysql
jupysql.init(ai_lab_config)

# AI_ANSWER

There are two UDfs for question answering. The `AI_ANSWER` udf uses some default configuration parameters and a preselected model. If you need more control over your prediction, the `AI_ANSWER_EXTENDED` udf allows you to set all configuration parameters.

We will start by showing you the `AI_ANSWER` udf.

Given the same question but two different contexts, we are using the `AI_ANSWER` UDF to extract some answers. In neither case the context has a direct answer to the question. We expect the answer to be relevant to the context.
                                                                                         This udf uses the text-generation pipeline with the [SmolLM2 model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) model for the prediction. Some of the answer content might be knowledge pulled from the model training data instead of the given context.

In [1]:
# This will be our question
TEST_QUESTION = 'What is bitumen used for?'

# Let's first try it first with the following context
TEST_CONTEXT1 = """
Apart from the stylish design features of new flat roofs, the other thing that’s moved on considerably is the technology
used to keep them weather-proof. Once flat roofs were notoriously prone to leaking and the problem could only be solved
with a boiling cauldron of tar. These days there are patch repair kits, liquid rubber membranes, and even quick,
efficient waterproofing paint that lasts for ages – and can even be applied in damp weather.
"""

# Make sure our texts can be used in an SQL statement.
TEST_QUESTION = TEST_QUESTION.replace("'", "''")
TEST_CONTEXT1 = TEST_CONTEXT1.replace("'", "''")

The udf takes various input parameters:

* question: The question text.
* context_text: The context text, associated with the question.

You need to supply these parameters in the correct order. Further information can be found in the  <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a>.


We will save the result in the variable `udf_output` to support automatic testing of this notebook.

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_ANSWER(
        '{{TEST_QUESTION}}',
        '{{TEST_CONTEXT1}}'
    )
)
SELECT answer, error_message FROM MODEL_OUTPUT

Let's now change the context and see a different answer.

In [ ]:
# New context
TEST_CONTEXT2 = """
You can make a wooden planter in a day, using treated timber. Simply work out how big an area you need,
cut the wood to size and follow our steps to putting the planter together. Make sure your wooden planter
has drainage holes, so plants don’t become waterlogged.
"""

# Make sure our text can be used in an SQL statement.
TEST_CONTEXT2 = TEST_CONTEXT2.replace("'", "''")

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_ANSWER(
        '{{TEST_QUESTION}}',
        '{{TEST_CONTEXT2}}'
    )
)
SELECT answer, error_message FROM MODEL_OUTPUT

The code above shows how the model works on a toy example. However, the main purpose of having a model deployed in the database is to get a quick response for a batch input. The performance gain comes from two factors - localization and parallelization. The first means that the input data never crosses the machine boundaries. The second means that multiple instances of the model are processing the data on all available nodes in parallel.

Another advantage of making predictions within the database is enhanced data security. The task of safeguarding privacy can be simplified given the fact that the source data never leaves the database machine.

In a more practical application, the question and the context would be stored in columns of a database table. For example, if we wanted to get an answer for each row of the input table `MY_TEXT_TABLE`, where the question is in the column `MY_QUESTION` and the context is in the column `MY_CONTEXT`, the SQL would look similar to this:
```
SELECT AI_ANSWER(MY_QUESTION, MY_CONTEXT) FROM MY_TEXT_TABLE;
```
Please note, that the response time observed on the provided example with a single input will not be scaled up linearly in case of multiple inputs. Much of the latency falls on loading the model into the CPU memory from BucketFS. This needs to be done only once regardless of the number of inputs.

# AI_ANSWER_EXTENDED

If you want to use a different model for prediction, you can use the `AI_ANSWER_EXTENDED` udf instead. This will work nearly the same, but you need to have a model installed and then set the udf parameters so the model can be found by the udf.

## Get a language model

To demonstrate the question-answering with the text-generation task, we will use the [openai-community/gpt2](https://huggingface.co/openai-community/gpt2) model.

We need to load the model from the Hugging Face Hub into the [BucketFS](https://docs.exasol.com/db/latest/database_concepts/bucketfs/bucketfs.htm). This could potentially be a long process, depending on the speed of the database connection. Unfortunately, we cannot tell exactly when it has finished. The notebook's hourglass may not be a reliable indicator. BucketFS will still be doing some work when the call issued by the notebook returns. Please wait for a few moments after that, before querying the model.

You might see a warning that some weights are newly initialized and the model should be trained on a down-stream task. Please ignore this warning. For the purpose of this demonstration, it is not important, the model should still be able to produce some meaningful output.

In [ ]:
from exasol.nb_connector.model_installation import install_model, TransformerModel
from transformers import AutoModelForCausalLM

# This is the name of the model at the Hugging Face Hub
MODEL_NAME = 'openai-community/gpt2'
install_model(ai_lab_config, TransformerModel(MODEL_NAME, 'text-generation', AutoModelForCausalLM))

## Use the language model

We are going to check the model output given the same question and contexts as before, but this time using the `AI_ANSWER_EXTENDED` UDF. 

The udf takes various input parameters:

* device_id: To run the UDF on a GPU, specify the valid cuda device ID.
* bucketfs_conn: The BucketFS connection name.
* sub_dir: The directory where the model is stored in the BucketFS.
* model_name: The name of the model to use for prediction.
* question: The question text.
* context_text: The context text, associated with the question.

You need to supply these parameters in the correct order. Further information can be found in the  <a href="https://github.com/exasol/transformers-extension/blob/main/doc/user_guide/user_guide.md" target="_blank" rel="noopener">User Guide</a>.


We will save the result in the variable `udf_output` to support automatic testing of this notebook.

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_ANSWER_EXTENDED(
        NULL,
        '{{ai_lab_config.bfs_connection_name}}',
        '{{ai_lab_config.bfs_model_subdir}}',
        '{{MODEL_NAME}}',
        '{{TEST_QUESTION}}',
        '{{TEST_CONTEXT1}}'
    )
)
SELECT answer, error_message FROM MODEL_OUTPUT

Let's now use the second context and see a different answer.

In [ ]:
%%sql --save udf_output
WITH MODEL_OUTPUT AS
(
    SELECT AI_ANSWER_EXTENDED(
        NULL,
        '{{ai_lab_config.bfs_connection_name}}',
        '{{ai_lab_config.bfs_model_subdir}}',
        '{{MODEL_NAME}}',
        '{{TEST_QUESTION}}',
        '{{TEST_CONTEXT2}}'
    )
)
SELECT answer, error_message FROM MODEL_OUTPUT